###Feature Engineering

In [ ]:
!pip install category_encoders

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.7/85.7 kB 2.0 MB/s eta 0:00:00


In [ ]:
# Trend/Delta Features
df = df.sort_values(by=['aircraft_id', 'flight_hours'])
df['temp_delta'] = df.groupby('aircraft_id')['engine_temp'].diff()
df['vibration_delta'] = df.groupby('aircraft_id')['vibration_level'].diff()
df['oil_pressure_delta'] = df.groupby('aircraft_id')['oil_pressure'].diff()

In [ ]:
#Fill NaNs from first diff
df[['temp_delta','vibration_delta','oil_pressure_delta']] = df[['temp_delta','vibration_delta','oil_pressure_delta']].fillna(0)

In [ ]:
#Stress/Interaction Features
df['cycles_per_hour'] = df['flight_cycles'] / (df['flight_hours'] + 1)
df['stress_score'] = df['vibration_level'] * (df['turbulence']+1) * (df['airport_condition']+1)

In [ ]:
#Maintenance interactions
df['maintenance_risk'] = df['last_maintenance_days'] * (1 - df['maintenance_score'])

In [ ]:
#Rolling averages (short-term trends)
df['engine_temp_roll3'] = df.groupby('aircraft_id')['engine_temp'].transform(lambda x: x.rolling(3, min_periods=1).mean())
df['vibration_roll3'] = df.groupby('aircraft_id')['vibration_level'].transform(lambda x: x.rolling(3, min_periods=1).mean())
df['oil_pressure_roll3'] = df.groupby('aircraft_id')['oil_pressure'].transform(lambda x: x.rolling(3, min_periods=1).mean())

In [ ]:
#Encode categoricals
from category_encoders.target_encoder import TargetEncoder

encoder = TargetEncoder(cols=['aircraft_id','route_type'])
df[['aircraft_id','route_type']] = encoder.fit_transform(df[['aircraft_id','route_type']], df['failure_within_30_days'])